# LangChain basics: one review, one step at a time

TrailMix sells outdoor gear. We will use **the same tent review** throughout.
First we ask Gemini for a reply. Then we add prompts, chains, and structured data.
Finally we use these pieces in the 10-review lab.

**Before starting:** finish the root README setup and put your Gemini key in `.env`.
Select **Session 8 (LangChain)**. Run each cell in order with **Shift+Enter**.
Imports appear when we first need them. You do not need to memorize them.

| Word | Meaning here |
| --- | --- |
| String (`str`) | Text, such as `"tent"` |
| Dictionary (`dict`) | Named values, such as `{"product": "tent"}` |
| List | Several values in order, such as `["tent", "boots"]` |
| Method | An action after a dot, such as `model.invoke(...)` |

LangChain supplies pieces we can connect. A piece that accepts input and returns
output is called a **Runnable**. We will see concrete examples before checking that name.


## 1. Call a model · slides 5–16


### 1a. Connect to Gemini

**Concept:** The model is the part that talks to Gemini. The key stays in your local .env file.

**Run:** Load the key and create the model connection. This cell does not send a request.

**Look for:** “Ready” means the Python setup completed; the next cell checks the API call.


In [ ]:
from dotenv import load_dotenv, find_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(find_dotenv())  # Read GOOGLE_API_KEY from .env.
MODEL_ID = "gemini-3.8-flash"  # Change the model here for this notebook.
model = init_chat_model(f"google_genai:{MODEL_ID}")
print("Ready")


### 1b. Send one review

**Concept:** invoke means “run once.” The reply is an AIMessage object; .text gives us its words.

**Run:** Ask for a short reply to our tent review. This makes one Gemini request.

**Look for:** AIMessage, followed by a short draft reply. Your wording may differ.


In [ ]:
review = "My two-person tent leaked during light rain on its first night."
reply = model.invoke("Write a short, polite support reply to: " + review)
print(type(reply).__name__)
print(reply.text)


**Pause:** `review` stores the input text. `reply` stores the object Gemini returned.
`reply.text` reads its text. LangChain converts the API's JSON response into that
Python object for us. Token counts and other metadata can wait until later.


### 1c. Separate the rules from the review

**Concept:** SystemMessage contains our instructions. HumanMessage contains the review. AIMessage is the reply.

**Run:** Give Gemini a role, then send the same review.

**Look for:** A brief reply that follows the system instruction. Change the tone and rerun to compare.


In [ ]:
from langchain.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage("You help TrailMix customers. Be polite. Use two short sentences."),
    HumanMessage(review),
]
reply = model.invoke(messages)
print(reply.text)


## 2. Fill a prompt template · slides 11, 17–18


### 2. Put the review into a named blank

**Concept:** A template is an instruction we can reuse. {review} is a blank, filled by the dictionary key "review".

**Run:** Create a chat prompt and print its filled messages. Nothing is sent to Gemini yet.

**Look for:** The system rule and the tent review inside the human message.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You help TrailMix customers. Write two polite sentences."),
    ("human", "Review: {review}"),
])
inputs = {"review": review}  # The key must match the blank.
filled_prompt = prompt.invoke(inputs)
print(filled_prompt.to_string())


**Try it:** replace the dictionary value with `"My hiking socks are warm and comfortable."`
and rerun the cell. The instructions stay the same, but the review changes.

If you type `"reveiw"` as the key, Python raises a `KeyError` because `"review"`
is missing. Restore the key to continue. Literal braces in a template use `{{` and `}}`.


## 3. Turn the reply into text · slides 13, 20


### 3a. Use a text parser

**Concept:** A parser converts a reply into a value our code can use. StrOutputParser returns a string.

**Run:** Parse the reply we already have. This makes no new Gemini request.

**Look for:** str, followed by the same text you saw in the reply.


In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
text = parser.invoke(reply)
print(type(text).__name__)
print(text)


### 3b. Join the three pieces

**Concept:** The pipe | passes the result on its left to the next piece on its right. The connected pieces form a chain.

**Run:** Fill the prompt, call Gemini, and get a string in one request.

**Look for:** A plain text reply. We no longer need to write .text after calling the chain.


In [ ]:
# Input dict -> filled prompt -> Gemini reply -> string
reply_chain = prompt | model | parser
answer = reply_chain.invoke({"review": review})
print(answer)


### 3c. Recognize a Runnable

**Concept:** The prompt, model, parser, and whole chain can all do a job. A message carries data between jobs.

**Run:** Check the objects we have already used. This cell makes no Gemini request.

**Look for:** Four True values, then False for a message.


In [ ]:
from langchain_core.runnables import Runnable

print(isinstance(prompt, Runnable))
print(isinstance(model, Runnable))
print(isinstance(parser, Runnable))
print(isinstance(reply_chain, Runnable))
print(isinstance(HumanMessage(review), Runnable))


If a chain fails, inspect it in three parts: fill the prompt, call the model, then
parse the reply. We already ran each of these steps above. Check the last value
that looked correct before changing anything.


## 4. Run once, run a list, or stream · slide 21


### 4a. Run a small batch

**Concept:** batch accepts a list of inputs and returns a list of results in the same order.

**Run:** Reply to two reviews. max_concurrency=1 handles one input at a time to keep the first batch gentle.

**Look for:** Two replies, in the same order as the two reviews.


In [ ]:
two_reviews = [
    {"review": review},
    {"review": "My hiking socks are warm and comfortable."},
]
answers = reply_chain.batch(two_reviews, config={"max_concurrency": 1})
for answer in answers:
    print(answer)
    print()


### 4b. Show text as it arrives

**Concept:** stream yields small pieces instead of waiting for the complete answer. Our text parser makes each piece a string.

**Run:** Stream a reply to the same tent review.

**Look for:** Text appearing in pieces. flush=True asks Python to display each piece immediately.


In [ ]:
for piece in reply_chain.stream({"review": review}):
    print(piece, end="", flush=True)
print()


**Choose:** `invoke` for one input, `batch` for several inputs, `stream` for visible
typing. All three still call the model. If you get a 429 error, pause and retry later.


## 5. Ask for a dictionary · slides 24–27


A reply such as “The customer sounds unhappy about a leaking tent” is easy to
read, but difficult to turn into CSV columns. We want named values instead:

```text
{"sentiment": "negative", "product": "two-person tent", "issue": "leaks in light rain"}
```

A **schema** describes these fields. Read the next dictionary from the outside in:

- `properties`: the field names and the kind of value each holds.
- `"type": "string"`: the value is text.
- `enum`: a fixed list of allowed answers.
- `required`: every result must include these field names.


### 5a. Describe the three fields

**Concept:** A JSON schema is a normal Python dictionary. We do not need to create a class.

**Run:** Define the fields for one review. This cell makes no request.

**Look for:** sentiment, product, and issue are all required. A review without a complaint uses issue="none".


In [ ]:
review_schema = {
    "title": "ReviewFacts",
    "description": "Facts from one product review.",
    "type": "object",
    "properties": {
        "sentiment": {"type": "string", "enum": ["positive", "neutral", "negative"]},
        "product": {"type": "string"},
        "issue": {"type": "string", "description": "Problem including any safety risk, or 'none'."},
    },
    "required": ["sentiment", "product", "issue"],
}


### 5b. Extract facts from our review

**Concept:** with_structured_output asks Gemini to follow the schema. Here it returns a dictionary instead of an AIMessage.

**Run:** Extract the facts, then read one value using its key.

**Look for:** A dictionary and its issue value. Good structure still needs a human check for correct facts.


In [ ]:
extractor = model.with_structured_output(review_schema, method="json_schema")
facts = extractor.invoke(review)
print(facts)
print(facts["issue"])


## 6. Run steps in order · slides 31, 33


### 6. Use extracted facts in the next prompt

**Concept:** A sequence runs one step after another. The second step needs the dictionary from the first step.

**Run:** Build a reply prompt with the same field names as our schema. Connect extraction to the reply chain.

**Look for:** A draft based on the extracted product and issue. This run makes two Gemini requests.


In [ ]:
facts_reply_prompt = ChatPromptTemplate.from_template(
    "Write two polite support sentences. Do not promise a refund. "
    "Sentiment: {sentiment}. Product: {product}. Issue: {issue}."
)
facts_reply_chain = facts_reply_prompt | model | StrOutputParser()
sequence = extractor | facts_reply_chain
print(sequence.invoke(review))


`extractor | facts_reply_chain` already creates a `RunnableSequence`.
`RunnableSequence(extractor, facts_reply_chain)` is another way to write it, after
importing `RunnableSequence` from `langchain_core.runnables`. Use the pipe in this lab.


## 7. Get a reply and urgency · slides 32–33


### 7a. Describe the urgency answer

**Concept:** Urgency has three allowed values. We call the field level to make it easy to read.

**Run:** Build a second schema and a prompt that reads the same facts as the reply prompt.

**Look for:** high for safety risks, medium for other problems, and low when there is no problem.


In [ ]:
urgency_schema = {
    "title": "Urgency", "description": "Support priority.", "type": "object",
    "properties": {"level": {"type": "string", "enum": ["low", "medium", "high"]}},
    "required": ["level"],
}
urgency_prompt = ChatPromptTemplate.from_template(
    "Choose high for injury or safety risk, medium for other problems, low for no problem. "
    "Sentiment: {sentiment}. Product: {product}. Issue: {issue}."
)
urgency_chain = urgency_prompt | model.with_structured_output(
    urgency_schema, method="json_schema"
)


### 7b. Give both branches the same facts

**Concept:** RunnableParallel sends the same input to two independent branches. It collects their answers under names we choose.

**Run:** Generate a reply and an urgency level from the facts saved in section 5.

**Look for:** reply is a string; urgency is a dictionary containing level. This makes two requests.


In [ ]:
from langchain_core.runnables import RunnableParallel

reply_and_urgency = RunnableParallel(
    reply=facts_reply_chain,
    urgency=urgency_chain,
)
draft = reply_and_urgency.invoke(facts, config={"max_concurrency": 2})
print(draft["reply"])
print(draft["urgency"]["level"])


**Read the two brackets:** `draft["urgency"]` selects the urgency branch's answer.
`["level"]` selects its low/medium/high value.

| Need | Use |
| --- | --- |
| Next step needs the previous answer | `a | b` (a sequence) |
| Two independent jobs use the same input | `RunnableParallel(...)` |
| Repeat a job for a list of inputs | `.batch(...)` |

## 8. Continue to the TrailMix lab · slides 35–38

Open `02-trailmix-reviews-lab/README.md`, then `lab.py`.
You will reuse the schema, reply chain, urgency chain, and batch method from here.
The file contains **10 reviews**. There are only three TODOs.

```text
10 reviews -> extract facts -> reply + urgency -> combine columns -> CSV
```

We use two visible batch calls in the lab so you can inspect the facts before
generating replies. The code comments explain the small amount of CSV handling.

**Five things to remember**

- The model calls Gemini and returns a reply.
- A prompt template fills named blanks.
- The pipe passes each result to the next step.
- A schema describes the dictionary fields we need.
- Parallel branches share input; batch repeats work for a list.

---

## Optional: other tools shown in the slides

You can stop here and do the lab. These short examples explain the remaining
slide names. Run them only if your instructor has time.


### Optional A. A text prompt and a history slot · slide 11

**Concept:** PromptTemplate creates one piece of text. MessagesPlaceholder inserts a list of earlier messages into a chat template.

**Run:** Format both kinds locally. Neither sends a Gemini request.

**Look for:** One filled text prompt, then a chat prompt with the earlier review in its history.


In [ ]:
from langchain_core.prompts import PromptTemplate, MessagesPlaceholder

plain = PromptTemplate.from_template("Summarize: {review}")
print(plain.invoke({"review": review}).to_string())
with_history = ChatPromptTemplate.from_messages([
    ("system", "Help TrailMix customers."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
print(with_history.invoke({
    "history": [HumanMessage(review)], "question": "What happened?"
}).to_string())


### Optional B. A list parser · slides 13, 25

**Concept:** CommaSeparatedListOutputParser splits comma-separated text into a Python list.

**Run:** Print its format instructions and parse a small text example locally.

**Look for:** A list containing tent, leak, and rain. No Gemini request is needed.


In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

list_parser = CommaSeparatedListOutputParser()
print(list_parser.get_format_instructions())
print(list_parser.invoke("tent, leak, rain"))


### Optional C. A JSON parser · slides 24–26

**Concept:** JsonOutputParser converts JSON text into a dictionary. Its format instructions ask for JSON, but do not enforce our field names.

**Run:** Request JSON using a prompt, model, and parser.

**Look for:** A dictionary. The lab uses native structured output when specific fields are required.


In [ ]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_prompt = ChatPromptTemplate.from_template(
    "Review: {review}\nReturn sentiment, product, and issue.\n{format}"
)
json_chain = json_prompt | model | json_parser
print(json_chain.invoke({
    "review": review, "format": json_parser.get_format_instructions()
}))


### Optional D. Catch a parsing problem · slides 24, 28

**Concept:** Extra chatter can break strict JSON reading. A LangChain parser also raises an error when its input is not parseable.

**Run:** Catch two errors using text examples. No Gemini request is made.

**Look for:** Two short error messages, while the notebook keeps running.


In [ ]:
import json
from langchain_core.exceptions import OutputParserException

try:
    json.loads('Here you go! ```json {"sentiment": "negative"} ```')
except json.JSONDecodeError:
    print("json.loads needs only valid JSON text.")
try:
    JsonOutputParser().invoke("This is not JSON.")
except OutputParserException:
    print("The output parser could not read this reply.")


### Optional E. Inspect a structured reply · slides 12, 28

**Concept:** include_raw returns the original message, the parsed dictionary, and any parsing error.

**Run:** Ask for the same facts and inspect the result. This makes one Gemini request.

**Look for:** parsed usually holds a dictionary and parsing_error is None. API failures can still raise an error.


In [ ]:
inspect_extractor = model.with_structured_output(
    review_schema, method="json_schema", include_raw=True
)
result = inspect_extractor.invoke(review)
print(result["parsed"])
print(result["parsing_error"])
print(result["raw"].usage_metadata)
